In [ ]:
from pathlib import Path

import sqlite3
import pandas as pd

# Set pandas option to display all columns
pd.set_option('display.max_columns', None)

db_path = Path.home() / "AppData/LocalLow/FlopMongers/Gunfish/stats.db"

print(db_path.resolve())

with sqlite3.connect(db_path) as conn:
    # Read the list of tables that exist in the database
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
    print("Tables in the database:")
    print(tables)


In [ ]:
# Query schemas of each table
with sqlite3.connect(db_path) as conn:
    for table_name in tables['name']:

        schema = pd.read_sql(f"PRAGMA table_info({table_name});", conn)
        print(f"\nSchema of table '{table_name}':")
        print(schema)


In [ ]:
# Number of .NET ticks between 0001-01-01 and 1970-01-01
DOTNET_TO_UNIX_TICKS = 621355968000000000

def convert_dates(df):
    for col in df.columns:
        if col.endswith("Time"):
            ticks = df[col].astype("Int64")

            # Convert .NET ticks → Unix nanoseconds
            unix_ns = (ticks - DOTNET_TO_UNIX_TICKS) * 100

            df[col] = pd.to_datetime(
                unix_ns,
                unit="ns",
                errors="coerce",
                utc=True,   # optional but recommended
            ).dt.floor("s")
    return df

In [ ]:
matches = convert_dates(pd.read_sql('SELECT * FROM MatchResult', sqlite3.connect(db_path)))
matches

In [ ]:
levels = convert_dates(pd.read_sql('SELECT * FROM LevelResult', sqlite3.connect(db_path)))
levels

In [ ]:
player_matches = convert_dates(pd.read_sql('SELECT * FROM PlayerMatchResult', sqlite3.connect(db_path)))
player_matches

In [ ]:
player_damage = convert_dates(pd.read_sql('SELECT * FROM PlayerDamage', sqlite3.connect(db_path)))
player_damage

In [ ]:
player_deaths = convert_dates(pd.read_sql('SELECT * FROM PlayerDamage WHERE IsFatal = 1', sqlite3.connect(db_path)))
player_deaths

In [ ]:
player_spawns = convert_dates(pd.read_sql('SELECT * FROM PlayerSpawn', sqlite3.connect(db_path)))
player_spawns

In [ ]:
query = Path("player-level-results.sql").read_text()
player_levels = convert_dates(pd.read_sql(query, sqlite3.connect(db_path)))
player_levels

In [ ]:
query = Path("player-lives.sql").read_text()
player_lives = convert_dates(pd.read_sql(query, sqlite3.connect(db_path)))
player_lives